# Reactant Optimization Workflow

This notebook prepares the ground-state structures used by BorylXAT-DB. It enumerates borane radicals, Lewis bases, and chlorinated substrates; generates xTB/CREST conformer-search inputs; prepares Gaussian optimization and single-point jobs; and parses completed calculations back into the curated CSV tables.

The heavy quantum-chemistry files are expected in an external calculation workspace (`root_file`). The public repository stores the curated summary tables and final database products, not every raw Gaussian/xTB output.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from DFTStructureGenerator import borane_xat_workflow, mol_manipulation
import os
import pandas as pd

In [ ]:
# Gaussian route sections used in this reactant-preparation notebook.
# OPT_METHOD is used for ground-state optimization/frequency calculations.
# SPE_METHOD and SPE_WFN_METHOD are used for higher-level single-point energies
# and wavefunction-based charge/spin-density post-processing.
OPT_METHOD = "opt freq b3lyp/6-31g(d) em=gd3bj scrf=(smd,solvent=toluene) nosymm"
SPE_METHOD = "wb97xd/6-311+g(d,p) scrf=(smd,solvent=toluene) nosymm"
SPE_WFN_METHOD = "wb97xd/6-311+g(d,p) scrf=(smd,solvent=toluene) nosymm output=wfn"


In [ ]:
# External calculation workspace for generated xTB/Gaussian files.
# This is intentionally separate from the repository's curated `Data/` folder.
root_file = "E:/work/B_Cl_Nu/Data"
if not os.path.isdir(root_file):
    os.mkdir(root_file)

# Main generated directories used by the reactant workflow.
mol_xtb_file = os.path.join(root_file, 'Mol_xtb')
if not os.path.isdir(mol_xtb_file):
    os.mkdir(mol_xtb_file)
mol_dir = os.path.join(root_file, 'Mols')
dft_dir = os.path.join(root_file, 'GS_OPT')
spe_dir = os.path.join(root_file, 'GS_SPE')


## Calculation Workspace Layout

`root_file` points to the external scratch/work directory used for generated xTB and Gaussian jobs. Update it before rerunning the workflow on another machine or HPC system.

Expected generated subdirectories include:

- `Mol_xtb_2/`: CREST/GFN2-xTB conformer-search inputs and outputs.
- `Mols2/`: RDKit `.mol` structures generated from the curated reactant tables.
- `GS_OPT2/`: Gaussian B3LYP-D3(BJ)/6-31G(d)/SMD(toluene) optimization and frequency jobs.
- `GS_SPE2/`: wB97X-D/6-311+G(d,p)/SMD(toluene) single-point corrections.
- `GS_SPE_wfn2/`: single-point jobs with wavefunction output for charge and spin-density post-processing.
- `dust_bin/` and `*_imp/`: failed logs and regenerated recovery inputs.


## 1. Enumerate Reactants and Reactive Sites


In [ ]:
# Regenerate the component tables and reactive-site annotations from the curated Excel file.
# This cell overwrites the CSV summaries below; skip it when using the committed CSV files directly.
duplicate_N_id, duplicate_Cl_id = borane_xat_workflow.generate_combinations(
    reactant_file='Data/csvs/Data_Structure_Update_Sum.xlsx',
    Bresult_file='Data/csvs/reactants_B.csv',
    Nresult_file='Data/csvs/reactants_N.csv',
    Clresult_file='Data/csvs/reactants_Cl.csv'
)


In [ ]:
# Persisted reactive-site duplicates from the curated reactant tables.
# Use these values when rerunning downstream cells without regenerating the Excel-derived CSVs.
duplicate_N_id = [9, 43, 285, 310, 314, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 372, 375, 376]
duplicate_Cl_id = [624, 625, 626, 627, 628, 629, 630, 631, 632, 633, 634, 635, 636,
       637, 638, 639, 640, 642, 644, 645, 652, 653, 654, 655, 656, 657, 658, 659, 660, 661, 662, 663, 664,
       665, 666, 667, 668, 669, 670, 671, 672, 673, 674, 675, 676, 677,
       678, 679, 680, 681, 682, 683, 684, 685, 686, 687, 688, 689, 690,
       691, 692, 693, 694, 695, 696, 697, 698, 699, 700, 701, 702, 703,
       704, 705, 706, 707, 708, 709, 710, 711, 713, 714, 716, 717, 718, 719, 720, 721, 722]


## 2. Generate xTB Conformer-Search Inputs


In [ ]:
# Generate CREST/GFN2-xTB conformer-search inputs for isolated borane radicals and Lewis bases.
borane_xat_workflow.B_N_Single_Xtb(
    root_file=root_file,
    Bresult_file='Data/csvs/reactants_B.csv',
    Nresult_file='Data/csvs/reactants_N.csv',
    mol_xtb_name='Mol_xtb_2'
)


In [ ]:
# Generate conformer-search inputs for chloride substrates and B-LB reactant/product complexes.
borane_xat_workflow.B_N_Cl_reactant_product_Xtb(
    root_file=root_file,
    Bresult_file='Data/csvs/reactants_B.csv',
    Nresult_file='Data/csvs/reactants_N.csv',
    Clresult_file='Data/csvs/reactants_Cl.csv',
    duplicate_N_id=duplicate_N_id,
    duplicate_Cl_id=duplicate_Cl_id,
    mol_xtb_name='Mol_xtb_2',
    mol_name='Mols2'
)


## 3. Generate Gaussian Optimization Inputs


In [ ]:
# Full production target list. Uncomment these two lines for a complete rebuild.
# all_strs = ['Cl_r', 'B_N_r', 'Cl_p', 'Cl_p_d', 'B_N_p', 'B_N_p_d', 'B_N_r_d', 'N_single']
# all_spins = [1, 2, 2, 2, 1, 1, 2, 1]

# Active rerun subset. This is currently restricted to chloride classes from the later cleanup pass.
all_strs = ['Cl_r', 'Cl_p_d', 'Cl_p']
all_spins = [1, 2, 2]

for name, spin in zip(all_strs, all_spins):
    mol_xtb_file = os.path.join(root_file, 'Mol_xtb_2')
    root_dir = os.path.join(mol_xtb_file, name)
    mol_dir = os.path.join(root_file, 'Mols2')
    dft_dir = os.path.join(root_file, 'GS_OPT2')
    if not os.path.isdir(dft_dir):
        os.mkdir(dft_dir)

    dft_dir_ = os.path.join(dft_dir, name)
    if not os.path.isdir(dft_dir_):
        os.mkdir(dft_dir_)

    borane_xat_workflow.smiles_DFT_calc(root_dir, mol_dir, dft_dir_, method=OPT_METHOD, conf_limit=10, SpinMultiplicity=spin)


### 3.1 Manual Recovery and Single-Point Cleanup

The cells below are selective rerun/recovery helpers. They should be executed only after checking which Gaussian jobs failed in the external calculation workspace.


In [ ]:
# Manual recovery for failed optimization logs.
# Verify `dft_dir`, `mol_dir`, and the target class before running this cell.
mol_manipulation.error_improve(dft_dir, mol_dir, 'Cl_r', 'dust_bin', improve_dir='Cl_r_imp')


In [ ]:
# Generate wB97X-D single-point inputs for recovered duplicate-site chloride radical products.
borane_xat_workflow.SPE_DFT_calc(root_file, 'GS_OPT2/Cl_p_d', 'GS_SPE2/Cl_p_d', 'Mols2', method=SPE_METHOD)


In [ ]:
# Switch parser/recovery helpers to the cleanup single-point directory.
spe_dir = os.path.join(root_file, 'GS_SPE2')


In [ ]:
# Manual recovery for failed duplicate-site chloride single-point jobs.
mol_manipulation.error_improve(spe_dir, mol_dir, 'Cl_p_d', 'dust_bin', improve_dir='Cl_p_d_imp')


In [ ]:
# Manual recovery for selected B-LB product single-point jobs from a cleanup batch.
mol_manipulation.error_improve(spe_dir, mol_dir, 'B_N_p_new', 'dust_bin', improve_dir='B_N_p_new_imp')


In [ ]:
# Generate wavefunction-enabled single-point inputs for Hirshfeld charge analysis.
borane_xat_workflow.SPE_DFT_calc_wfn(root_file, 'GS_OPT2/Cl_p_d', 'GS_SPE_wfn2/Cl_p_d', 'Mols2', method=SPE_WFN_METHOD)


In [ ]:
# Generate wavefunction-enabled single-point inputs for isolated Lewis bases.
borane_xat_workflow.SPE_DFT_calc_wfn(root_file, 'GS_OPT/N_single', 'GS_SPE_wfn/N_single', method=SPE_WFN_METHOD)


In [ ]:
# Generate wavefunction-enabled single-point inputs for duplicate-site B-LB product complexes.
borane_xat_workflow.SPE_DFT_calc_wfn(root_file, 'GS_OPT/B_N_p_d', 'GS_SPE_wfn/B_N_p_d', method=SPE_WFN_METHOD)


## 4. Parse Completed Jobs into Curated Tables

Run these cells after the relevant optimization, single-point, and wavefunction jobs have finished. They update the CSV files used by the database-building and modeling notebooks.


In [ ]:
# Parse completed monomer/chloride Gaussian jobs into reactant summary CSVs.
borane_xat_workflow.collection_dft_single(
    Bresult_path='Data/csvs/reactants_B.csv',
    Nresult_path='Data/csvs/reactants_N.csv',
    Clresult_path='Data/csvs/reactants_Cl.csv',
    mol_dir=mol_dir,
    dft_dir=dft_dir,
    spe_dir=spe_dir,
    duplicate_Cl_id=duplicate_Cl_id,
)


In [ ]:
# Parse completed B-LB reactant-complex jobs.
borane_xat_workflow.collection_dft_couple(
    Bresult_path='Data/csvs/reactants_B.csv',
    Nresult_path='Data/csvs/reactants_N.csv',
    B_N_result_path='Data/csvs/reactants_B_N.csv',
    type_='r',
    mol_dir=mol_dir,
    dft_dir=dft_dir,
    spe_dir=spe_dir,
    duplicate_N_id=duplicate_N_id,
)


In [ ]:
# Parse completed B-LB chloroborane product-complex jobs.
borane_xat_workflow.collection_dft_couple(
    Bresult_path='Data/csvs/reactants_B.csv',
    Nresult_path='Data/csvs/reactants_N.csv',
    B_N_result_path='Data/csvs/reactants_B_N.csv',
    type_='p',
    mol_dir=mol_dir,
    dft_dir=dft_dir,
    spe_dir=spe_dir,
    duplicate_N_id=duplicate_N_id,
)


In [ ]:
# Keep only successfully parsed B-LB complexes and chloride radical products.
data_csv = pd.read_csv('Data/csvs/reactants_B_N.csv')
data_csv = data_csv.dropna(subset=['deltaG_comb(kcal)', 'deltaG_react'])
data_csv.to_csv('Data/csvs/reactants_B_N.csv', index=False)

data_csv = pd.read_csv('Data/csvs/reactants_Cl.csv')
data_csv = data_csv.dropna(subset=['G_energy_p'])
data_csv.to_csv('Data/csvs/reactants_Cl.csv', index=False)


In [ ]:
# Recalculate chloride dechlorination free energies after cleanup filtering.
data_csv = pd.read_csv('Data/csvs/reactants_Cl.csv')
data_csv = data_csv.dropna(subset=['G_energy_r', 'G_energy_p'])
data_csv['deltaG_react'] = data_csv['G_energy_p'] - data_csv['G_energy_r']
data_csv.to_csv('Data/csvs/reactants_Cl.csv', index=False)
